#**Sprint 10 - Consumo de APIs REST, Métodos HTTP y Procesamiento con Pandas**

En este sprint se aborda la interacción con servicios web mediante solicitudes HTTP con la biblioteca `requests`, la interpretación sistemática de códigos de estado y la etructuración de datos en DataFrames de `pandas`para el análisis espacial y climático de Barcelona.

**Dependecnias iniciales**

In [16]:
import requests
import pandas as pd

#Ajustes de visualización
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)

##**Nivel 1: Exploración con API de laboratorio (JSONPlaceholder)**
###En este bloque se comprueba el ciclo de vida del protocolo HTTP (`GET`, `POST`, `PATCH`, `DELETE`) y el tratamiento de códigos de estado frente a un servicio simulado.

**1 y 2. Consulta GET de recursos base y métricas de respuesta**

Se conultan los tres recursos solicitados(`/posts`, `/users`, `/todos`), validando el código de respuesta `200 OK`y calculando la cantidad total de registros tras deserializar el cuerpo JSON.

In [17]:
API_LAB_URL = "https://jsonplaceholder.typicode.com"

endpoints = {
    "Publicaciones (/posts)": f"{API_LAB_URL}/posts",
    "Usuarios (/users)": f"{API_LAB_URL}/users",
    "Tareas (/todos)": f"{API_LAB_URL}/todos"
}

respuestas_laboratorio = {}

for nombre, url in endpoints.items():
    res = requests.get(url)
    respuestas_laboratorio[nombre] = res
    datos = res.json()
    print(f"Recurso: {nombre}")
    print(f"  - Código HTTP: {res.status_code}")
    print(f"  - Cantidad total de registros: {len(datos)}")
    print("-" * 45)

Recurso: Publicaciones (/posts)
  - Código HTTP: 200
  - Cantidad total de registros: 100
---------------------------------------------
Recurso: Usuarios (/users)
  - Código HTTP: 200
  - Cantidad total de registros: 10
---------------------------------------------
Recurso: Tareas (/todos)
  - Código HTTP: 200
  - Cantidad total de registros: 200
---------------------------------------------


**3. Petición a recurso no existente (Error 404)**

Se solicita un recurso intencionadamente fuera de rango para verificar la respuesta `404 Not Found`

In [18]:
url_no_valida = f"{API_LAB_URL}/posts/sensor_invalido_9999"
res_404 = requests.get(url_no_valida)

print(f"Ruta consultada: {url_no_valida}")
print(f"Código de estado devuelto: {res_404.status_code} (Not Found)")

Ruta consultada: https://jsonplaceholder.typicode.com/posts/sensor_invalido_9999
Código de estado devuelto: 404 (Not Found)


**4. Creación ficticia mediante POST**

Se genera una solicitud `POST`enviando un diccionario codificado en JSON con los campos requeridos (`title`, `body`, `userID`). el servidor simula la persistencia devolciendo un código `201 Created`y el identificador generado.

In [19]:
registro_alerta = {
    "title": "Anomalía térmica en superficie - Eixample",
    "body": "Temperatura superficial crítica registrada en zona con baja reflectancia y escaso arbolado.",
    "userId": 12
}

res_post = requests.post(f"{API_LAB_URL}/posts", json=registro_alerta)

print(f"Código de estado: {res_post.status_code} (Created)")
print("Cuerpo de la respuesta JSON:")
print(res_post.json())

Código de estado: 201 (Created)
Cuerpo de la respuesta JSON:
{'title': 'Anomalía térmica en superficie - Eixample', 'body': 'Temperatura superficial crítica registrada en zona con baja reflectancia y escaso arbolado.', 'userId': 12, 'id': 101}


**5. Modificación parcial mediante PATCH**

Se actualiza únicamente un atributo de un registro existente sin sobreescribir la entidad completa. El servidor confirma la modificación con un código `200 OK`

In [20]:
url_post_1 = f"{API_LAB_URL}/posts/1"
modificacion = {
    "title": "Protocolo de contingencia frente a ola de calor activado"
}

res_patch = requests.patch(url_post_1, json=modificacion)

print(f"Código de estado: {res_patch.status_code} (OK)")
print("Objeto resultante con la modificación parcial aplicada:")
print(res_patch.json())

Código de estado: 200 (OK)
Objeto resultante con la modificación parcial aplicada:
{'userId': 1, 'id': 1, 'title': 'Protocolo de contingencia frente a ola de calor activado', 'body': 'quia et suscipit\nsuscipit recusandae consequuntur expedita et cum\nreprehenderit molestiae ut ut quas totam\nnostrum rerum est autem sunt rem eveniet architecto'}


**6. Eliminación de un recurso mediante DELETE**

Se lanza una petición `DELETE`contra la publicación identificada con ID 1. El código `200 OK`indica que la solicitud fue aceotada por el servidor de pruebas.

In [21]:
res_delete = requests.delete(url_post_1)

print(f"Código de estado: {res_delete.status_code} (OK)")
print("Cuerpo JSON devuelto tras la eliminación:", res_delete.json())

Código de estado: 200 (OK)
Cuerpo JSON devuelto tras la eliminación: {}


---
##**Nivel 2: Interacción con una API pública real**

###**API Seleccionada: NASA POWER API (Prediction Of Worldwide Energy Resources)**

### Para este nivel se emplea la NASA POWER API, una interfaz pública desarroalda por la NASA para suminstrar obser

## Documentación técnica de la API

* **URL Base:** `https://power.larc.nasa.gov/api`

### Endpoints identificados
1. `GET /temporal/daily/point`: Suministra agregaciones diarias de variables climáticas y de radiación solar para coordenadas de latitud/longitud dadas.
2. `GET /temporal/climatology/point`: Suministra promedios climatológicos multianuales para estudios históricos de referencia.

### Filtros y parámetros opcionales utilizados
* `latitude` y `longitude`: Coordenadas geográficas decimales (centro urbano de Barcelona: `41.3879`, `2.1699`).
* `parameters`: Parámetros físicos satelitales a extraer:
  * `T2M_MAX`: Temperatura máxima diaria a 2 metros de superficie (°C).
  * `T2M_MIN`: Temperatura mínima diaria a 2 metros (°C).
  * `ALLSKY_SFC_SW_DWN`: Radiación solar global descendente incidente en superficie ($MJ/m^2/día$).
* `community`: Sector de aplicación (`RE` para *Renewable Energy / Sustainable Cities*).
* `start` y `end`: Rango temporal en formato `YYYYMMDD`.
* `format`: Especificación explícita del formato estructurado (`JSON`).

### Formato de respuesta
* **Estructura:** JSON jerárquico bajo el estándar **GeoJSON** con los valores indexados por fecha.

**Extracción de datos térmicos satelitales (NASA POWER)**

In [22]:
url_nasa_power = "https://power.larc.nasa.gov/api/temporal/daily/point"

parametros_nasa = {
    "latitude": 41.3879,
    "longitude": 2.1699,
    "parameters": "T2M_MAX,T2M_MIN,ALLSKY_SFC_SW_DWN",
    "community": "RE",
    "start": "20230701",
    "end": "20230731",
    "format": "JSON"
}

res_nasa = requests.get(url_nasa_power, params=parametros_nasa)
print(f"Código de estado de la consulta a NASA: {res_nasa.status_code}")

datos_nasa = res_nasa.json()

# Inspección clara de metadatos devueltos
info_coordenadas = datos_nasa["geometry"]["coordinates"]
descripcion_parametros = datos_nasa["parameters"]

print(f"Coordenadas analizadas (Longitud, Latitud, Altitud): {info_coordenadas}")
print("\nVariables satelitales procesadas:")
for var, desc in descripcion_parametros.items():
    print(f"  - {var}: {desc['longname']} ({desc['units']})")

Código de estado de la consulta a NASA: 200
Coordenadas analizadas (Longitud, Latitud, Altitud): [2.17, 41.388, 302.35]

Variables satelitales procesadas:
  - T2M_MAX: Temperature at 2 Meters Maximum (C)
  - T2M_MIN: Temperature at 2 Meters Minimum (C)
  - ALLSKY_SFC_SW_DWN: All Sky Surface Shortwave Downward Irradiance (kW-hr/m^2/day)


**Conversión de las series térmicas satelitales a DataFrame**

Aplanamos el diccionario anidado dentro de `properties.parameter`para estructurar una serie temporal limpia donde cada fila represente un día del mes con sus métricas térmicas.

In [23]:
series_parametros = datos_nasa["properties"]["parameter"]

# Construimos el DataFrame con las fechas como índice
df_calor_satelital = pd.DataFrame(series_parametros)
df_calor_satelital.index.name = "fecha"
df_calor_satelital = df_calor_satelital.reset_index()

# Convertimos la cadena de texto a formato fecha
df_calor_satelital["fecha"] = pd.to_datetime(df_calor_satelital["fecha"], format="%Y%m%d")

# Renombramos columnas para claridad analítica
df_calor_satelital.columns = ["fecha", "temp_max_c", "temp_min_c", "radiacion_solar_mj_m2"]

print(f"Dimensiones del DataFrame térmico: {df_calor_satelital.shape}")
df_calor_satelital.head(10)

Dimensiones del DataFrame térmico: (31, 4)


,fecha,temp_max_c,temp_min_c,radiacion_solar_mj_m2
0,2023-07-01,29.67,19.16,7.6764
1,2023-07-02,30.54,19.66,7.2245
2,2023-07-03,31.02,20.06,6.2489
3,2023-07-04,28.58,21.25,6.3199
4,2023-07-05,28.09,20.36,4.9176
5,2023-07-06,26.94,20.15,6.0578
6,2023-07-07,29.82,20.02,6.1171
7,2023-07-08,34.29,20.57,7.8646
8,2023-07-09,34.20,21.51,7.4750
9,2023-07-10,33.84,22.41,7.6606


---
##**Nivel 3: API de Open Data Barcelona**

###**Dataset Seleccionado: Red de Refugios Climáticos de Barcelona (Xarxa de Refugis Climàtics)**

### Para conectar directamente con la hipótesis de mitigación y equidad de tu proyecto, se consulta el portal CKAN de Barcelona buscando la red de refugios climáticos (espacios interiores y exteriores acondicioandos para proteger a colectivos vulnerables durante picos de calor).

**1. Localización del dataset mediante `package_search**

Buscamos datasets bajo la clave **"refugis climatics"** e identificamos el nombre de sistema (**name**)

In [24]:
CKAN_BCN_URL = "https://opendata-ajuntament.barcelona.cat/data/api/action"

headers_consulta = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) UrbanHeatResearch/1.0"
}

# 1. Búsqueda de datasets vinculados a refugios climáticos
params_busqueda = {"q": "refugis climatics", "rows": 5}
res_search = requests.get(f"{CKAN_BCN_URL}/package_search", params=params_busqueda, headers=headers_consulta)
print(f"Código de estado de la búsqueda: {res_search.status_code}")

conjuntos_datos = res_search.json()["result"]["results"]

for item in conjuntos_datos:
    print(f"• Dataset: {item['title']} | ID técnico: {item['name']}")

# Tomamos el primer dataset coincidente
dataset_seleccionado = conjuntos_datos[0]["name"]

Código de estado de la búsqueda: 200
• Dataset: Climate shelters network in the city of Barcelona | ID técnico: xarxa-refugis-climatics


**2. Inspección con `package_show` y selección de recurso con DataStore activo**

Consultamos la ficha técnica del dataset con `package_show`para recuperar su lista de recursos. Se filtra por aquellos en formto CSV o JSON cuyo campo `datastore_active` sea **True**, requisito para poder efecturar consultas analíticas mediante la API de CKAN.

In [25]:
res_show = requests.get(f"{CKAN_BCN_URL}/package_show", params={"id": dataset_seleccionado}, headers=headers_consulta)
print(f"Código de estado package_show: {res_show.status_code}")

recursos_dataset = res_show.json()["result"]["resources"]
resource_id_valido = None

for rec in recursos_dataset:
    formato = rec.get("format", "").upper()
    es_datastore = rec.get("datastore_active", False)

    # Comprobamos formato válido y base de datos activa en API
    if es_datastore and formato in ["CSV", "JSON"]:
        resource_id_valido = rec["id"]
        print("Recurso válido localizado:")
        print(f"  - Nombre: {rec.get('name')}")
        print(f"  - Formato: {formato}")
        print(f"  - Resource ID: {resource_id_valido}")
        break

if not resource_id_valido:
    raise RuntimeError("No se encontró ningún recurso con datastore activo en el dataset seleccionado.")

Código de estado package_show: 200
Recurso válido localizado:
  - Nombre: opendatabcn_NP-NASIA_xarxa-refugis-climatics-csv.csv
  - Formato: CSV
  - Resource ID: 7ecae024-6cb2-427d-b2d0-e170500e2a38


**3, 4 y 5. Extracción con `datastore_search` (mínimo 100 registros), DataFrame y guardado a CSV**

Se lanza la consulta mediante `datastore_search`fijando un límite de 150 registros para superar el umbral mínimo exigido, se transforma la respuesta en un DataFrame y se exporta a `.csv`

In [26]:
# 3. Petición datastore_search con limit >= 100
params_datastore = {
    "resource_id": resource_id_valido,
    "limit": 150
}

res_datastore = requests.get(f"{CKAN_BCN_URL}/datastore_search", params=params_datastore, headers=headers_consulta)
print(f"Código de estado de la consulta a datos: {res_datastore.status_code}")

registros_recibidos = res_datastore.json()["result"]["records"]

# 4. Transformación en DataFrame de pandas
df_refugios_bcn = pd.DataFrame(registros_recibidos)
print(f"Volumen de registros capturados: {len(df_refugios_bcn)}")

print("\nPrimeras filas del DataFrame de refugios urbanos:")
display(df_refugios_bcn.head())

# 5. Guardado en disco local
archivo_csv_salida = "refugios_climaticos_bcn.csv"
df_refugios_bcn.to_csv(archivo_csv_salida, index=False, encoding="utf-8")
print(f"\nArchivo generado y guardado correctamente: '{archivo_csv_salida}'")

Código de estado de la consulta a datos: 200
Volumen de registros capturados: 150

Primeras filas del DataFrame de refugios urbanos:


,addresses_roadtype_name,addresses_end_street_number,geo_epgs_4326_lon,values_attribute_name,addresses_road_name,values_category,addresses_zip_code,values_value,addresses_town,timetable,addresses_district_name,geo_epgs_25831_x,addresses_start_street_number,register_id,institution_id,addresses_main_address,addresses_district_id,addresses_roadtype_id,addresses_type,addresses_neighborhood_id,_id,name,addresses_road_id,created,geo_epgs_25831_y,institution_name,modified,geo_epgs_4326_lat,values_description,values_id,addresses_neighborhood_name,values_outstanding,values_attribute_id
0,,None,2.129771989264385,Web,Plaça de Comas,Informació d'interès,8028,http://ajuntament.barcelona.cat/lescorts,Barcelona,"<table class=""timetable-table""><tr class=""time...",Les Corts,427240.3762247957,18,﻿99400762003,92086015374,True,4,,,19,1,Refugi climàtic,84907,2025-07-25T14:25:47.701608,4581933.959648992,Seu del Districte de les Corts,2026-08-06T09:57:19.461365,41.38564399418199,,680520,les Corts,True,100003
1,,167,2.161172547087871,Web,Carrer de Tamarit,Informació d'interès,8015,https://laalgodonera.net/,Barcelona,"<table class=""timetable-table""><tr class=""time...",Eixample,429858.603,167,﻿99400788727,None,True,2,,,10,2,Botiga La Algodonera,329902,2026-10-08T08:46:55.354204,4581164.898,,2026-10-08T08:46:55.354204,41.3789498562006,,758845,Sant Antoni,False,100003
2,,32,2.1635616297656908,Instagram,Carrer del Comte Borrell,Informació d'interès,8015,https://www.instagram.com/laperrueco/,Barcelona,"<table class=""timetable-table""><tr class=""time...",Eixample,430056.072,32,﻿99400788728,None,True,2,,,10,3,La Perru Eco.Lògica,87009,2026-10-08T08:49:03.585985,4580924.222,,2026-10-08T08:51:55.793454,41.376799352781035,,758850,Sant Antoni,False,100008
3,,None,2.179464430011531,Text per a la web de Refugis Climàtics - detalls,C Sots-tinent Navarro,Variables,8002,"<p><span class=""ql-cursor"">﻿</span>Previsió de...",BARCELONA,"<table class=""timetable-table""><tr class=""time...",Ciutat Vella,431392.4119315087,12,﻿99400780973,99400609279,True,1,,,2,4,Espai amb Jocs d'Aigua,328706,2026-05-08T11:19:50.250702,4581599.957776578,Jardins de la Plaça Carmé Simó,2026-08-03T11:31:10.858794,41.38300055778059,,732473,el Barri Gòtic,False,100097
4,,4,2.1590431196628765,Web,Carrer de Bonavista,Informació d'interès,8012,https://bonavista-apartments.com/ca,Barcelona,,Gràcia,429700.842,4,﻿99400780855,None,True,6,,,31,5,Oficina de Bonavista Apartments,51103,2026-05-07T09:48:59.940093,4583258.353,,2026-08-25T13:45:11.358440,41.39779112106451,,732169,la Vila de Gràcia,False,100003



Archivo generado y guardado correctamente: 'refugios_climaticos_bcn.csv'
